<div style="border-top:5px solid #0D9488;padding:28px 30px;background:#F8FAFC;border-radius:0 0 14px 14px;color:#0F172A;">
<p style="font-size:12px;letter-spacing:3px;color:#0D9488;font-weight:700;margin:0 0 12px;">RAG LAB / THE EXECUTABLE GUIDE</p>
<h1 style="font-size:38px;line-height:1.15;margin:0 0 16px;">From documents to<br>answers you can verify.</h1>
<p style="font-size:17px;max-width:760px;line-height:1.6;margin:0;">A guided walkthrough of ingestion, retrieval, grounded generation, adaptive search, and end-to-end tracing.</p>
<p style="margin:20px 0 0;font-size:13px;">REMOTE POSTGRESQL + OPENSEARCH &nbsp; / &nbsp; LOCAL OLLAMA &nbsp; / &nbsp; HOSTED LANGFUSE</p>
</div>

![Document passages flow through retrieval and local inference into an answer with evidence tabs, connected to two remote storage systems.](../docs/assets/rag-lab-hero.png)

*Documents become answers through retrieval, local inference, and inspectable evidence.*

This notebook uses the project's actual modules. The default **offline run** loads the fictional sample corpus and exercises chunking, rank fusion, and evidence assembly without credentials. **Live mode** uses your configured services for embeddings, ingestion, retrieval, answers, and evaluation.

[Notebook instructions](README.md) ? [Documentation home](../docs/README.md) ? [Implementation](../docs/implementation.md)

## The learning path

| Phase | Question to answer | Sections |
| --- | --- | --- |
| Connect | What runs locally, and what is hosted? | 01?02 |
| Ingest | How does a document become retrievable chunks? | 03?06 |
| Retrieve | Which evidence reaches the model, and why? | 07?09 |
| Generate | Can I verify the answer against its sources? | 10?11 |
| Measure | Is retrieval useful, and where does time go? | 12?14 |

**Before a live run:** configure the project `.env`, run local Ollama, and install both model names. The [setup guide](../docs/getting-started.md) covers the connection settings. PostgreSQL and OpenSearch are already hosted; this notebook never starts containers.

**Read the outputs accurately:** RRF examples are explicitly illustrative. A live generation cell reports an actual model response only when live mode is enabled. The source notebook contains no saved live output or credentials.

## 01 ? Find the project and prepare the workspace

The notebook locates the repository from its current directory, imports its modules, and creates a few display helpers. It does not read `.env` in offline mode. If imports fail, select the project's `.venv` kernel or run the notebook launch command from its README.

In [ ]:
import html
import json
import math
import os
from pathlib import Path

from IPython.display import HTML, Markdown, display

from rag_lab.chunking import chunk_text
from rag_lab.config import Settings
from rag_lab.loaders import load_file
from rag_lab.models import AskRequest, Hit, SearchRequest
from rag_lab.pipeline import RagService, build_context
from rag_lab.search import reciprocal_rank_fusion

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "src/rag_lab").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the RAG Lab project directory.")
os.chdir(ROOT)


def table(columns, rows):
    headings = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in row) + "</tr>" for row in rows
    )
    display(
        HTML(
            '<div style="overflow-x:auto;margin:16px 0;">'
            '<table style="width:100%;text-align:left;border-collapse:collapse;">'
            f'<thead style="background:#F1F5F9;">{headings}</thead>'
            f"<tbody>{body}</tbody></table></div>"
        )
    )


def note(message):
    display(
        HTML(
            '<div style="border-left:3px solid #0D9488;background:#F8FAFC;'
            'padding:12px 16px;margin:12px 0;color:#0F172A;">' + html.escape(message) + "</div>"
        )
    )


note("Project located. The application modules are ready; no services have been contacted.")

### Choose the run mode

Leave the defaults for an offline Run All. For a full live pass, set `RUN_LIVE=True` and `INGEST_SAMPLE=True`; add initialization only when needed. The sample-write flag inserts the fictional corpus into your configured remote stores. Evaluation makes additional model-embedding and search requests. Restart the kernel after changing `.env`.

In [ ]:
RUN_LIVE = False
INITIALIZE_REMOTE = False
INGEST_SAMPLE = False
RUN_EVALUATION = False

service = None
live_ready = False
answer = None
agentic_answer = None
trace_ids = []

table(
    ["Control", "Enabled"],
    [
        ["Live service connections", RUN_LIVE],
        ["Initialize application tables/index", INITIALIZE_REMOTE],
        ["Ingest the fictional sample corpus", INGEST_SAMPLE],
        ["Run retrieval evaluation", RUN_EVALUATION],
    ],
)

## 02 ? Connect and check readiness

Only live mode constructs `Settings` and `RagService`. Initialization creates application resources without deleting existing documents. A health check before initialization may report PostgreSQL `UndefinedTable` and an OpenSearch missing-index error.

The table below shows service state rather than connection strings. Langfuse authentication is checked when it is enabled; this is distinct from proving that a trace arrived at the remote server.

![Two-zone architecture showing Python and Ollama locally, with independent connections to remote PostgreSQL, OpenSearch, and optional Langfuse.](../docs/assets/rag-lab-architecture.png)

*Python coordinates local Ollama inference with remote canonical storage, search, and optional hosted tracing.*

In [ ]:
if RUN_LIVE:
    try:
        settings = Settings(_env_file=ROOT / ".env")
        service = RagService(settings)
        if INITIALIZE_REMOTE:
            service.initialize()
        report = service.health(check_tracing=True)
    except Exception as error:
        if service is not None:
            service.close()
            service = None
        raise RuntimeError(
            f"Live setup failed ({type(error).__name__}). Check .env and run rag doctor. "
            "Connection details are omitted."
        ) from None
    table(
        ["Dependency", "Status", "Diagnostic"],
        [
            [name, value["status"], value.get("error_type", value.get("cluster_status", ""))]
            for name, value in report["services"].items()
        ],
    )
    live_ready = all(value["status"] == "ok" for value in report["services"].values())
    note("Langfuse state: " + report["tracing"]["status"])
    if not live_ready:
        note(
            "Live experiments are paused. Initialize missing resources or fix the dependency. "
            "Then rerun this cell."
        )
else:
    note("Offline mode: no .env loaded, no service clients created, and no network requests made.")

## 03 ? Meet the knowledge corpus

The three sample documents describe **fictional Northstar policies**. Each loader returns a title, stable source URI, and extracted text. PostgreSQL later preserves this text so the search index can be rebuilt without reopening the original file.

Local files use their absolute URI as the document identity. The display below uses only filenames to keep the walkthrough portable.

In [ ]:
corpus = []
for path in sorted((ROOT / "data/sample").glob("*.md")):
    title, source, text = load_file(path)
    corpus.append({"path": path, "title": title, "source": source, "text": text})
assert len(corpus) == 3, "Expected the three bundled sample documents."
table(
    ["Document", "Characters", "Opening"],
    [[item["path"].name, len(item["text"]), item["text"].splitlines()[0]] for item in corpus],
)
backup = next(item for item in corpus if item["title"] == "backup-policy")
display(Markdown(backup["text"]))

## 04 ? Experiment with overlapping chunks

A chunk is a unit of retrieval. Smaller windows make boundaries easier to see in this small corpus. The production defaults are 1,400 characters with a requested 200-character overlap; this experiment uses 420 and 80.

Change either setting and rerun the cell. Observe how the number of chunks changes and which sentences cross boundaries. These are **character windows, not tokenizer limits**. Whitespace is trimmed and an end boundary may move to a nearby space.

![A document is divided into three passage cards, with repeated teal segments highlighting adjacent overlap.](../docs/assets/rag-lab-chunking.png)

*Repeating a small part of adjacent passages helps preserve context at a chunk boundary. This project measures its windows in characters.*

In [ ]:
WINDOW_CHARS = 420
OVERLAP_CHARS = 80

chunks = chunk_text(backup["text"], WINDOW_CHARS, OVERLAP_CHARS)
assert chunks and all(0 < len(text) <= WINDOW_CHARS for text in chunks)
table(
    ["Chunk position", "Characters", "First words", "Last words"],
    [[position, len(text), text[:85], text[-85:]] for position, text in enumerate(chunks)],
)
for position, text in enumerate(chunks):
    display(
        HTML(
            '<div style="margin:12px 0;padding:16px;border:1px solid #CBD5E1;border-radius:10px;">'
            f'<strong style="color:#0D9488;">Chunk {position}</strong>'
            f'<pre style="white-space:pre-wrap;line-height:1.6;">{html.escape(text)}</pre></div>'
        )
    )

> **Checkpoint:** find the retention rule in the displayed chunks. Would one chunk contain enough context to answer both ?How long?? and ?Who approves a restore?? Try a larger window and compare.

[Read the chunker](../src/rag_lab/chunking.py) ? [Understand document revisions](../docs/implementation.md#2-split-text-into-retrievable-units)

## 05 ? Encode meaning with local Ollama

Both documents and questions must use the same embedding configuration. Nomic's document/query task prefixes are applied inside the client. The live experiment shows vector shape and norm rather than dumping hundreds of values.

The configured dimension validates the model's native output; it does not resize a vector. A different model or prefix needs a fresh index and a complete reindex, even if the dimension is unchanged.

In [ ]:
if live_ready:
    document_vectors = service.ollama.embed([chunks[0]])
    query_vectors = service.ollama.embed(["How long are daily backups retained?"], purpose="query")
    table(
        ["Input type", "Model", "Dimensions", "Vector norm"],
        [
            [
                kind,
                settings.ollama_embed_model,
                len(vector),
                round(math.sqrt(sum(x * x for x in vector)), 4),
            ]
            for kind, vector in [("document", document_vectors[0]), ("query", query_vectors[0])]
        ],
    )
else:
    note("Embedding calls are skipped offline. In live mode, inspect real local vectors here.")

## 06 ? Ingest through the actual pipeline

Ingestion writes versioned text/vector records to OpenSearch, then commits the canonical document and active chunks in PostgreSQL. Search results later have to resolve to active SQL chunk IDs. Older search revisions are cleaned up after the SQL commit.

Enable `INGEST_SAMPLE` with live mode to store the three bundled documents. Unchanged revisions are skipped. The notebook's earlier chunk experiment does not change the configured ingestion settings.

In [ ]:
if live_ready and INGEST_SAMPLE:
    ingestion_results = [service.ingest_file(item["path"]) for item in corpus]
    table(
        ["Document", "Status", "Chunks", "Cleanup pending"],
        [
            [
                item["path"].name,
                result["status"],
                result.get("chunks", "unchanged"),
                result.get("cleanup_pending", False),
            ]
            for item, result in zip(corpus, ingestion_results, strict=True)
        ],
    )
    trace_ids.extend(result["trace_id"] for result in ingestion_results if result.get("trace_id"))
elif live_ready:
    note("Sample writes are off. Set INGEST_SAMPLE=True if these documents are not already stored.")
else:
    note("Offline mode does not write to PostgreSQL or OpenSearch.")

## 07 ? Compare keyword, semantic, and hybrid search

Use the same question for all three modes. Keyword retrieval matches terms; semantic retrieval compares meaning; hybrid retrieval fuses their ranks. Inspect which source appears and what its passage actually says.

This experiment calls `RagService.search`, including canonical SQL hydration. It does not generate an answer. Scores from different modes have different meanings and should not be compared as probabilities.

![A question splits into BM25 and vector retrieval branches whose ranked passages converge into one evidence list.](../docs/assets/rag-lab-hybrid-search.png)

*Hybrid retrieval combines keyword and semantic result ranks, rather than adding their raw scores.*

In [ ]:
QUESTION = "How long are daily backups retained, and who approves a production restore?"
retrieval_results = {}
if live_ready:
    for mode in ("keyword", "semantic", "hybrid"):
        hits = service.search(SearchRequest(query=QUESTION, mode=mode, top_k=5))
        retrieval_results[mode] = hits
        display(Markdown(f"### {mode.capitalize()} results"))
        table(
            ["Rank", "Source title", "Chunk", "Score", "Excerpt"],
            [
                [rank, hit.title, hit.position, round(hit.score, 5), hit.text[:240]]
                for rank, hit in enumerate(hits, 1)
            ],
        )
        if not hits:
            note(
                "No active chunks returned. Confirm sample ingestion "
                "before interpreting retrieval quality."
            )
else:
    note(
        "Live retrieval is skipped. The next section calculates RRF "
        "from an explicitly illustrative ranking."
    )

## 08 ? Calculate reciprocal rank fusion

BM25 and vector scores use different scales. RRF combines **positions**, using:

```text
score(chunk) = sum(1 / (60 + rank_in_each_list))
```

The lists below are an **illustrative example**, not output from OpenSearch. The backup chunk appears in both rankings and should rise above candidates seen in only one. Change the lists and rerun the cell.

In [ ]:
keyword_ranking = [{"_id": "backup"}, {"_id": "access"}]
semantic_ranking = [{"_id": "incident"}, {"_id": "backup"}]
fused = reciprocal_rank_fusion([keyword_ranking, semantic_ranking])
assert fused[0]["_id"] == "backup"
expected_backup_score = 1 / 61 + 1 / 62
assert math.isclose(fused[0]["_score"], expected_backup_score)
table(
    ["Combined rank", "Illustrative chunk", "RRF score"],
    [[rank, hit["_id"], f"{hit['_score']:.6f}"] for rank, hit in enumerate(fused, 1)],
)
note(
    "Agreement across rankings raises the backup chunk. "
    "The values are rank weights, not confidence estimates."
)

## 09 ? Assemble a bounded evidence context

The project numbers each included excerpt and budgets its serialized JSON, including escaped characters and metadata. If necessary, it truncates the final passage. Returned source text matches what the answer model actually received.

This offline experiment constructs `Hit` objects directly from the bundled backup text. The IDs and scores are demonstration values; **this is not a retrieval result**. Lower the budget and inspect which evidence still fits. The budget covers evidence JSON, not the question, instructions, or answer tokens.

![An answer card cites two numbered source excerpts while a third supplied excerpt remains uncited.](../docs/assets/rag-lab-evidence.png)

*Numbered excerpts make a generated claim easier to check. Supplied evidence still needs human verification.*

In [ ]:
context_hits = [
    Hit(
        chunk_id=f"demo-backup-{position}",
        document_id="demo-backup",
        title=backup["title"],
        source=backup["source"],
        position=position,
        text=text,
        score=1 / (61 + position),
    )
    for position, text in enumerate(chunks)
]
EVIDENCE_BUDGET = 1000
context_json, context_sources = build_context(context_hits, EVIDENCE_BUDGET)
assert len(context_json) <= EVIDENCE_BUDGET
assert all(
    item["text"] == source.text
    for item, source in zip(json.loads(context_json), context_sources, strict=True)
)
table(
    ["Citation", "Title", "Chunk position", "Included characters"],
    [
        [f"[{source.citation}]", source.title, source.position, len(source.text)]
        for source in context_sources
    ],
)
print(f"Evidence JSON: {len(context_json)} / {EVIDENCE_BUDGET} characters")
print(json.dumps(json.loads(context_json), indent=2))

## 10 ? Ask, then verify the evidence

The live call retrieves passages and asks the local chat model to answer using them. Inspect the answer, then read the supplied excerpts. A source being supplied does not prove every generated claim is supported or that the source was cited.

No generated answer is fabricated for the offline preview. With empty evidence, the application returns a deterministic abstention without calling the answer generator.

In [ ]:
if live_ready:
    answer = service.ask(AskRequest(query=QUESTION, mode="hybrid", top_k=5))
    display(Markdown(answer.answer))
    table(
        ["Model", "Elapsed ms", "Sources", "Trace ID"],
        [
            [
                answer.model,
                answer.elapsed_ms,
                len(answer.sources),
                answer.trace_id or "disabled / unavailable",
            ]
        ],
    )
    for source in answer.sources:
        display(Markdown(f"### [{source.citation}] {source.title} ? chunk {source.position}"))
        print(source.text)
    if answer.trace_id:
        trace_ids.append(answer.trace_id)
else:
    note(
        "Answer generation is skipped offline. Enable live mode "
        "to receive an actual model answer and its evidence."
    )

## 11 ? Observe a bounded adaptive workflow

Ask an unsupported question. In agentic mode, Ollama grades the excerpts; if none are useful, it rewrites the search query once, retrieves again, and grades against the original question. The pipeline then generates or abstains after at most **two retrieval attempts**.

Model relevance decisions can be wrong. The expected behavior for this fictional corpus is an acknowledgment that cafeteria policy is unsupported. The steps expose actions and counts, not private model reasoning.

![A two-row workflow shows retrieve and grade, followed by generation or one rewrite and retry, then generation or abstention.](../docs/assets/rag-lab-agentic.png)

*Relevance grading can trigger one query rewrite. The workflow ends after at most two retrieval attempts.*

In [ ]:
if live_ready:
    agentic_answer = service.ask(
        AskRequest(
            query="What is the Northstar office cafeteria menu next Tuesday?",
            mode="hybrid",
            top_k=5,
            agentic=True,
        )
    )
    display(Markdown(agentic_answer.answer))
    table(
        ["Step", "Action", "Detail"],
        [[number, step.action, step.detail] for number, step in enumerate(agentic_answer.steps, 1)],
    )
    assert sum(step.action == "retrieve" for step in agentic_answer.steps) <= 2
    if agentic_answer.trace_id:
        trace_ids.append(agentic_answer.trace_id)
else:
    note(
        "Adaptive retrieval is skipped offline. "
        "Inspect RagService._ask to locate the two-attempt bound."
    )

## 12 ? Evaluate retrieval before judging prose

The bundled dataset has one expected document title per question. We check whether that title appears among the top five **chunks**, then deduplicate titles to calculate its document rank and reciprocal rank.

- **Document recall@5 chunks:** fraction of queries whose expected title appears in those five chunks.
- **Mean reciprocal document rank:** average of `1 / rank`, with zero when the expected document is absent.

The three questions are a learning exercise, not a production benchmark. The metrics below are measured only when live evaluation is enabled; the offline preview reports no quality scores. Different corpora or extra documents can change the rankings.

In [ ]:
evaluation_examples = json.loads((ROOT / "data/evaluation.json").read_text(encoding="utf-8"))
assert evaluation_examples, "The evaluation dataset must not be empty."
if live_ready and RUN_EVALUATION:
    metric_rows = []
    for mode in ("keyword", "semantic", "hybrid"):
        found = []
        reciprocal_ranks = []
        for example in evaluation_examples:
            hits = service.search(SearchRequest(query=example["query"], mode=mode, top_k=5))
            titles = list(dict.fromkeys(hit.title for hit in hits))
            rank = (
                titles.index(example["expected_title"]) + 1
                if example["expected_title"] in titles
                else None
            )
            found.append(rank is not None)
            reciprocal_ranks.append(1 / rank if rank else 0)
        metric_rows.append(
            [
                mode,
                len(evaluation_examples),
                round(sum(found) / len(found), 3),
                round(sum(reciprocal_ranks) / len(reciprocal_ranks), 3),
            ]
        )
    table(
        ["Mode", "Queries", "Document recall@5 chunks", "Mean reciprocal document rank"],
        metric_rows,
    )
else:
    table(
        ["Dataset", "Questions", "Status"],
        [["Fictional Northstar policies", len(evaluation_examples), "Not evaluated in this run"]],
    )
    note(
        "Set RUN_LIVE=True and RUN_EVALUATION=True after ingesting "
        "the sample corpus to measure retrieval."
    )

## 13 ? Follow the operation in Langfuse

An ingestion and a later question are separate traces. Within a question, child observations share the parent `rag.ask` trace. Inspect retrieval, query embeddings, SQL hydration, grading/rewrite when used, context assembly, and generation.

Ollama's reported input/output token counters populate generation observations; embedding input usage is recorded when available. Content capture defaults to off. Enabling it sends questions, prompts, excerpts, answers, and source paths to the configured Langfuse host.

A trace ID identifies an observation created locally; **confirm arrival in the Langfuse UI**. An authentication check does not verify OTLP event delivery. The next cell closes clients and drains buffered observations before you check the final trace.

![A conceptual trace waterfall nests search and embedding spans under retrieval, followed by context assembly and generation.](../docs/assets/rag-lab-tracing.png)

*One question becomes a parent trace with nested observations for retrieval, context, and model generation. The illustration shows conceptual timing, not measured execution.*

In [ ]:
unique_trace_ids = list(dict.fromkeys(trace_ids))
if unique_trace_ids:
    table(
        ["Operation trace", "ID"],
        [[number, trace_id] for number, trace_id in enumerate(unique_trace_ids, 1)],
    )
    note("Locate these IDs in your Langfuse project's trace view after running cleanup below.")
else:
    note(
        "No trace IDs were collected in this run. Offline mode creates no telemetry; "
        "live mode also needs tracing enabled."
    )

## 14 ? Close the experiment and choose the next one

Run cleanup after live work, including after an earlier cell fails. Shutdown drains buffered tracing events; a force-killed kernel may lose pending observations. Restart the kernel to begin a clean experiment with changed configuration.

In [ ]:
if service is not None:
    service.close()
    service = None
live_ready = False
note("Experiment complete. Any application clients created in this run are closed.")

### Continue with a deliberate experiment

| Change | Observe | Keep fixed |
| --- | --- | --- |
| Chunk size and overlap | Boundary quality, chunk count, evidence completeness | The document and question |
| Retrieval mode | Whether the expected passage is retrieved | Corpus, top-k, and question |
| Candidate pool or top-k | Recall, context size, and latency | Embedding model and evaluation set |
| Agentic mode | Relevance decisions, retry path, abstention | The unsupported question |
| Langfuse text capture | Visibility of prompts versus operational metadata | The RAG request |

Changing live chunking settings requires reingestion/reindexing before comparing retrieval. Changing embedding identity requires a fresh index and a complete reindex. Do not compare quality results from differently indexed corpora as if only the query changed.

**You can now trace the path:** source text ? overlapping chunks ? embeddings and search ? active SQL evidence ? bounded context ? cited answer ? observations.

[Read the implementation](../docs/implementation.md) ? [Practice the lessons](../docs/lessons.md) ? [Trace the pipeline](../docs/observability.md)